## EEG Segmentation from .xdf file
This notebook is responsible for dividing one EEG session into its segments, meaning that each segment corresponds to one genre listens to. The input for this is an .xdf file from an EEG session. The output will be a directory with new .xdf files each representing one song of the whole session

## General setup

In [ ]:
# Setup path to data directory
PATH_TO_DATA = "../../data"
SONG_COUNT = 50
SESSION_NUMBER = "S005"

In [ ]:
table = {}
# now here comes the tricky part: we have to store everything in a dictionary
# how do we want our data structured?
# as rows, we want the song index
rows = []
for i in range(SONG_COUNT):
  rows.append(f"Song{i}")

In [ ]:
# Imports
import pyxdf
import numpy as np
import pandas as pd
import os
from scipy.io import savemat

In [ ]:
def setup(filename):
  data, header = pyxdf.load_xdf(f"{PATH_TO_DATA}/{filename}")
  return data, header

In [ ]:
data, header = setup("sub-P001_ses-S005_task-Default_run-001_eeg.xdf")

In [ ]:
# print important info
for stream in data:
  print(f"Stream {stream["info"]["name"]}")

In [ ]:
def find_stream(stream_name):
  for stream in data:
    if stream["info"]["name"] == [stream_name]:
      print(f"Found the stream {stream_name}!")
      return stream
  print("Couldn't find the stream.")

In [ ]:
# Get all the streams
eeg_stream = find_stream("Headset_EEG_DR250")
music_markers_stream = find_stream("MusicMarkers")
song_stream = find_stream("Song")
temperature_stream = find_stream("Headset_TMP_DR1")
ppg_algo_stream = find_stream("Headset_PPG_ALGO_DR25")
imu_stream = find_stream("Headset_IMU_DR30")

In [ ]:
def get_channel_names(biosignal_stream):
    ret = []
    acc = map(lambda x: x["label"][0], biosignal_stream["info"]["desc"][0]["channels"][0]["channel"])
    for channel_name in acc:
        ret.append(channel_name)
    return ret

In [ ]:
# Read out channel names of each stream
# EEG Channels
print("EEG Channels:")
eeg_channels = get_channel_names(eeg_stream)
for elem in eeg_channels:
    print(elem)
print("===========")
print("IMU Channels:")
imu_channels = get_channel_names(imu_stream)
for elem in imu_channels:
    print(elem)
print("===========")
print("Temperature Channels:")
temperature_channels = get_channel_names(temperature_stream)
for elem in temperature_channels:
    print(elem)
print("===========")
print("PPG Channels:")
ppg_channels = get_channel_names(ppg_algo_stream)
for elem in ppg_channels:
    print(elem)

In [ ]:
# Get timestamps and data
# Annotation: data holds the real values; timestamps the corresponding time the specific value was measured at
eeg_data = eeg_stream["time_series"]
eeg_timestamps = eeg_stream["time_stamps"]

temperature_data = temperature_stream["time_series"]
temperature_timestamps = temperature_stream["time_stamps"]

ppg_algo_data = ppg_algo_stream["time_series"]
ppg_algo_timestamps = ppg_algo_stream["time_stamps"]

imu_data = imu_stream["time_series"]
imu_timestamps = imu_stream["time_stamps"]


music_markers_data = music_markers_stream["time_series"]
music_markers_timestamps = music_markers_stream["time_stamps"]

song_data = song_stream["time_series"]
song_timestamps = song_stream["time_stamps"]

In [ ]:
# the genres are also columns which we can already initialize
table["genre"] = []
for i in range(len(song_data)-1):
  table["genre"].append(song_data[i][1])

## Common Helper functions

In [ ]:
def getStartFromSegment(start, timestamps, prev_pointer):
    start_pointer = prev_pointer
    while True:
      if timestamps[start_pointer] >= start: # found the start timestamp or rather got the closest one
        break
      start_pointer += 1
    print(f"The Start pointer is {start_pointer}")
    return start_pointer

In [ ]:
def segment_signal_data(timestamp_list, signal_list, channel_count):
  # Read out EEG channel data
  data_acc = [] # stores data over the course of all channels and each song i guess idk
  pointer = 0 # pointer that iterates over all the timestamps/data
  for i in range(0, SONG_COUNT*2, 2):
    # get the start and end timestamp of the song
    print(f"======START {i}=====")
    start = music_markers_timestamps[i]
    print(f"Start timestamp: {start}")
    end = music_markers_timestamps[i+1]
    print(f"End timestamp: {end}")

    # find the index in which the timestamp can be found
    pointer = getStartFromSegment(start, timestamp_list, pointer)
    print(f"The pointer is {pointer}")
    pointer_buf = pointer # store the pointer since it is increased steadily

    print("START READING OUT SIGNAL DATA")
    song_data = [] # the eeg data from this specific song (song i)
    for j in range(channel_count):
      print(f"Channel {j} is being read out")
      channel_data = [] # signal data from the channel that is being read out rn
      current_channel = signal_list[:, j] # get the current channel
      while timestamp_list[pointer] < end:
        # append data to the channel data list
        channel_data.append(current_channel[pointer])
        pointer += 1
      pointer = pointer_buf # reset the pointer
      song_data.append(channel_data) # append the channel data to the current song's signal data
    data_acc.append(song_data)
    print(f"======END {i}=====")
  return data_acc

In [ ]:
def store_in_table(columns, segmented_signal_data):
  # initialize the rest of the columns in the table
  for channel in columns:
    table[channel] = []

  # now we have to construct the actual values of the columns
  for song_counter in range(SONG_COUNT):
    current_song = segmented_signal_data[song_counter]
    counter = 0
    for channel_data in current_song:
      column_name = columns[counter]
      counter += 1
      table[column_name].append(channel_data)

In [ ]:
def debug_print_shapes(segmented_signal_data):
  for i in range(SONG_COUNT):
    print(f"This is the shape of song {i+1}: {np.shape(segmented_signal_data[i])}")

## EEG Data

In [ ]:
# Remove the 10 other channels
eeg_data_processed = eeg_data[:, :8]

In [ ]:
np.shape(eeg_timestamps)

In [ ]:
channel_data_acc = segment_signal_data(timestamp_list=eeg_timestamps, signal_list=eeg_data_processed, channel_count=8)

In [ ]:
# Print all shapes of the eeg data
debug_print_shapes(channel_data_acc)

In [ ]:
# as columns, we obviously want the data. the column names should suggest the name of the Channel. to keep it simple, we can just have <BIOSIGNAL><CHANNEL_NAME>
eeg_channels_col = [f"EEG_{eeg_channels[i]}" for i in range(8)]
print(eeg_channels_col)
store_in_table(columns=eeg_channels_col, segmented_signal_data=channel_data_acc)

## Other Biosignals

### Temperature

In [ ]:
temperature_data_buffer = segment_signal_data(timestamp_list=temperature_timestamps, signal_list=temperature_data, channel_count=3) 

In [ ]:
debug_print_shapes(temperature_data_buffer)

In [ ]:
temp_columns = [f"TEMP_{temperature_channels[i]}" for i in range(3)]
print(temp_columns)
store_in_table(columns=temp_columns, segmented_signal_data=temperature_data_buffer)

### IMU Data

In [ ]:
np.shape(imu_data)

In [ ]:
imu_data_buffer = segment_signal_data(timestamp_list=imu_timestamps, signal_list=imu_data, channel_count=6)

In [ ]:
debug_print_shapes(imu_data_buffer)

In [ ]:
imu_columns = [f"IMU_Ch{imu_channels[i]}" for i in range(6)]
store_in_table(imu_columns, imu_data_buffer)

In [ ]:
print(imu_columns)

### PPG Data

In [ ]:
np.shape(ppg_algo_data)

In [ ]:
ppg_data_buffer = segment_signal_data(timestamp_list=ppg_algo_timestamps, signal_list=ppg_algo_data, channel_count=23)

In [ ]:
debug_print_shapes(ppg_data_buffer)

In [ ]:
ppg_columns = [f"PPG_{ppg_channels[i]}" for i in range(23)]
print(ppg_columns)
store_in_table(ppg_columns, ppg_data_buffer)

## Construct final table and file saving

In [ ]:
df = pd.DataFrame(table)
df.index = rows
df

In [ ]:
# Save the table
os.makedirs(f"{PATH_TO_DATA}/signal_data", exist_ok=True)

# create a folder for numpy data
os.makedirs(f"{PATH_TO_DATA}/signal_data/numpy", exist_ok=True)
np.save(f"{PATH_TO_DATA}/signal_data/numpy/{SESSION_NUMBER}.npy", df.to_numpy())
# create a folder for matlab data
os.makedirs(f"{PATH_TO_DATA}/signal_data/matlab", exist_ok=True)
savemat(f"{PATH_TO_DATA}/signal_data/matlab/{SESSION_NUMBER}.mat", table)

os.makedirs(f"{PATH_TO_DATA}/signal_data/csv", exist_ok=True)
df.to_csv(f"{PATH_TO_DATA}/signal_data/csv/{SESSION_NUMBER}.csv")
